In [ ]:
"""
AA-CBR evaluation for the 2D pipeline using GT slot targets + OS survival labels.

Uses ground-truth slot targets derived from BraTS PNG seg masks, 
and bins Survival_days from BraTS_OS.csv as the outcome.  
This gives a theoretical upper-bound on what the 2D characterisation
features can predict against a real clinical endpoint.
"""

import argparse
import csv
import re
import sys
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

cwd = Path.cwd()
print("Current working directory:", cwd)
root_path = Path("/path/to/BrainWear_Kareem")
project_root = root_path / "FYP"
print("Project root:", project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from datasets.brats2020_png import BraTS2020PNGDataset, batch_seg_to_slot_targets_2d
from aacbr.aacbr_parallel import AACBRParallel
from aacbr.configs.brats_model_config import BraTSOutcomeConfig
from utils.characterisations import (
    TumourCharacterisationLarge2D,
    TumourCharacterisationSmall2D,
    TumourCharacterisationSmall2DV2,
    TumourCharacterisationLarge2DV2,
)

CHAR_MODELS = {
    "small":    TumourCharacterisationSmall2D,
    "large":    TumourCharacterisationLarge2D,
    "small_v2": TumourCharacterisationSmall2DV2,
    "large_v2": TumourCharacterisationLarge2DV2,
}

In [ ]:
def group_by_patient(dataset: BraTS2020PNGDataset) -> dict[str, list[int]]:
    groups: dict[str, list[int]] = {}
    for i, (t2_path, _) in enumerate(dataset.samples):
        pid = Path(t2_path).parent.name
        groups.setdefault(pid, []).append(i)
    return groups


def build_patient_features(
    dataset: BraTS2020PNGDataset,
    groups: dict[str, list[int]],
    char_model,
    num_slots: int,
    device: torch.device,
    agg_mode: str = "sum",   # "sum" | "max"
) -> dict[str, np.ndarray]:
    """Aggregate per-slice GT characterisation vectors into one per-patient vector.

    sum: accumulates counts across slices (original behaviour)
    max: takes element-wise max — binary result meaning "present in any slice"
    """
    feats: dict[str, np.ndarray] = {}
    with torch.no_grad():
        for pid, idxs in tqdm(groups.items(), desc=f"Characterising (GT, {agg_mode})"):
            acc = char_model.default_case().astype(np.int64)
            for i in idxs:
                _t2, seg = dataset[i]
                slots = batch_seg_to_slot_targets_2d(seg.unsqueeze(0), num_slots)[0]
                v = char_model.characterisation_transform(slots).astype(np.int64)
                if agg_mode == "max":
                    acc = np.maximum(acc, v)
                else:
                    acc = acc + v
            feats[pid] = acc
    return feats


def _dedup(feats: np.ndarray, out: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    unique, inverse = np.unique(feats, axis=0, return_inverse=True)
    dedup_out = np.array([
        int(np.round(np.median(out[inverse == i])))
        for i in range(len(unique))
    ])
    return unique, dedup_out


def run_aacbr(
    train_feats: np.ndarray,
    train_out: np.ndarray,
    test_feats: np.ndarray,
    test_out: np.ndarray,
    char_model,
    cfg: BraTSOutcomeConfig,
    n_bins: int,
    *,
    strategy: str = "ordinal",
    strict: bool = True,
    use_supports: bool = False,
    smart_default: bool = False,
    dedup: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    ls_fn = lambda a, b: char_model.less_specific(a, b, strict=strict)

    def make_model(k):
        default_out = (1 if k == 0 else 0) if smart_default else cfg.default_outcome
        return AACBRParallel(
            less_specific=ls_fn,
            default_case=char_model.default_case(),
            default_outcome=default_out,
            include_supports=use_supports,
            supported_attack_chain=use_supports,
        )

    models = {k: make_model(k) for k in range(n_bins)}

    if strategy == "ordinal":
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o >= k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = binary[:, 1:].sum(axis=1)

    elif strategy == "flat":
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o == k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = np.array([
            max(np.flatnonzero(binary[i] == 1), default=cfg.default_class)
            for i in range(len(test_feats))
        ])

    elif strategy == "tournament_flat":
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] == k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        unclassified = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[unclassified])
            newly = np.zeros(len(test_feats), dtype=bool)
            newly[unclassified] = preds_k == 1
            preds[newly] = k
            unclassified[newly] = False

    elif strategy == "tournament_tree":
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] > k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        at_node = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[at_node])
            left_mask = np.zeros(len(test_feats), dtype=bool)
            left_mask[at_node] = preds_k == 0
            preds[left_mask] = k
            at_node[left_mask] = False

    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")

    return test_out, preds


def report(tag: str, y_true: np.ndarray, y_pred: np.ndarray, n_bins: int) -> dict:
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score,
        confusion_matrix, classification_report,
    )
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mae     = np.abs(y_true.astype(float) - y_pred.astype(float)).mean()
    labels  = list(range(n_bins))
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    report_dict = classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3, output_dict=True)
    macro = report_dict.get("macro avg", {})
    print(f"\n=== {tag} ===")
    print(f"Accuracy: {acc:.4f}   Balanced Acc: {bal_acc:.4f}   Ordinal MAE: {mae:.4f}")
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3))
    return {
        "accuracy":          float(acc),
        "balanced_accuracy": float(bal_acc),
        "f1":                float(macro.get("f1-score", 0.0)),
        "precision":         float(macro.get("precision", 0.0)),
        "recall":            float(macro.get("recall", 0.0)),
        "ordinal_mae":       float(mae),
        "confusion_matrix":  cm.tolist(),
        "per_class": {
            k: {m: v for m, v in vs.items() if m != "support"}
            for k, vs in report_dict.items()
            if isinstance(vs, dict)
        },
    }

In [ ]:
def _parse_survival_days(raw: str) -> float | None:
    """Parse a Survival_days cell, handling 'ALIVE (N days later)' entries."""
    raw = raw.strip()
    try:
        return float(raw)
    except ValueError:
        m = re.search(r"(\d+)", raw)
        if m:
            return float(m.group(1))
        return None


def load_os_csv(
    csv_path: str,
    n_bins: int = 5,
    resection_filter: str | None = None,
) -> tuple[dict[str, int], np.ndarray]:
    """Parse BraTS_OS.csv and bin Survival_days into ordinal classes."""
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pid = row["Brats20ID"].strip()
            days = _parse_survival_days(row["Survival_days"])
            resection = row["Extent_of_Resection"].strip()
            if days is None:
                continue
            if resection_filter and resection != resection_filter:
                continue
            rows.append((pid, days))

    if not rows:
        raise ValueError("No usable rows found in OS CSV after filtering.")

    pids = [r[0] for r in rows]
    survival = np.array([r[1] for r in rows])

    quantiles = np.quantile(survival, np.linspace(0, 1, n_bins + 1)[1:-1])
    bins = np.digitize(survival, quantiles)  # 0..n_bins-1

    print(f"OS CSV: {len(rows)} patients with survival data.")
    print(f"Survival days  min={survival.min():.0f}  median={np.median(survival):.0f}  max={survival.max():.0f}")
    print(f"Quantile thresholds ({n_bins} bins): {np.round(quantiles).astype(int).tolist()}")
    print(f"Class distribution: {np.bincount(bins, minlength=n_bins).tolist()}")

    return {pid: int(b) for pid, b in zip(pids, bins)}, quantiles

In [ ]:
from itertools import product as iproduct
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED = 0
TRAIN_FRAC = 0.8
NUM_SLOTS = 5
AGG_MODES = ["sum", "max"]
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = str(root_path / "Processed_BraTS2020_TrainingData_PNG_new_norm")
OS_CSV   = str(Path("/path/to/BrainWear_Kareem") / "BraTS_OS.csv")
CONFIG   = str(project_root / "aacbr" / "configs" / "brats_configs" / "flat_config.json")

cfg    = BraTSOutcomeConfig.from_json(CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset    = BraTS2020PNGDataset(data_dir=DATA_DIR, is_train=False)
all_groups = group_by_patient(dataset)

# -------------------------------------------------------------------
# Phase 1: Pre-build features once per (char_model, n_bins, agg_mode)
# -------------------------------------------------------------------
print("Pre-building features for all model/bin/agg combinations...")
feature_cache: dict[tuple, tuple] = {}

for char_name, n_bins, agg_mode in iproduct(
    ["small", "large", "small_v2", "large_v2"], [2, 3, 4, 5], AGG_MODES
):
    char_model = CHAR_MODELS[char_name]
    print(f"\n--- char={char_name} ({char_model.DIMS}D), n_bins={n_bins}, agg={agg_mode} ---")
    patient_to_class, _ = load_os_csv(OS_CSV, n_bins=n_bins)

    matched_pids   = sorted(pid for pid in all_groups if pid in patient_to_class)
    matched_groups = {pid: all_groups[pid] for pid in matched_pids}

    feats    = build_patient_features(dataset, matched_groups, char_model, NUM_SLOTS, device, agg_mode=agg_mode)
    features = np.array([feats[p] for p in matched_pids])
    outcomes = np.array([patient_to_class[p] for p in matched_pids])

    unique_count = len(np.unique(features, axis=0))
    print(f"  {len(matched_pids)} patients | Unique feature vectors: {unique_count} / {len(features)}")

    idx          = np.arange(len(matched_pids))
    class_counts = np.bincount(outcomes, minlength=n_bins)
    if np.any(class_counts < 2):
        rng     = np.random.default_rng(SEED)
        perm    = rng.permutation(len(idx))
        n_train = int(round(TRAIN_FRAC * len(idx)))
        train_idx, test_idx = perm[:n_train], perm[n_train:]
    else:
        train_idx, test_idx = train_test_split(
            idx, train_size=TRAIN_FRAC, random_state=SEED, stratify=outcomes
        )

    print(f"  Train: {len(train_idx)} | Test: {len(test_idx)}")
    feature_cache[(char_name, n_bins, agg_mode)] = (features, outcomes, train_idx, test_idx)

# -------------------------------------------------------------------
# Phase 2: Sweep all strategy × char × n_bins × agg × strict combinations
# -------------------------------------------------------------------
STRATEGIES = ["ordinal", "flat", "tournament_flat", "tournament_tree"]
n_combos   = len(feature_cache) * len(STRATEGIES) * 2  # strict True/False
print(f"\n\nRunning AA-CBR sweep ({n_combos} configurations)...")
sweep_results = []

for (char_name, n_bins, agg_mode), strategy, strict in iproduct(
    feature_cache.keys(), STRATEGIES, [True, False]
):
    features, outcomes, train_idx, test_idx = feature_cache[(char_name, n_bins, agg_mode)]
    char_model = CHAR_MODELS[char_name]

    y_true, y_pred = run_aacbr(
        features[train_idx], outcomes[train_idx],
        features[test_idx],  outcomes[test_idx],
        char_model, cfg, n_bins,
        strategy=strategy, strict=strict, use_supports=False,
        smart_default=False, dedup=True,
    )
    acc = float(accuracy_score(y_true, y_pred))
    mae = float(np.abs(y_true.astype(float) - y_pred.astype(float)).mean())
    sweep_results.append(dict(
        char=char_name, n_bins=n_bins, agg=agg_mode, strategy=strategy, strict=strict,
        acc=acc, mae=mae, y_true=y_true, y_pred=y_pred,
    ))

sweep_results.sort(key=lambda r: (-r["acc"], r["mae"]))

# -------------------------------------------------------------------
# Phase 3: Summary table and full report for the best config
# -------------------------------------------------------------------
print(f"\n{'char':<12} {'agg':<5} {'bins':>4} {'strategy':<16} {'strict':>6} {'Acc':>8} {'MAE':>8}")
print("-" * 70)
for r in sweep_results:
    print(
        f"{r['char']:<12} {r['agg']:<5} {r['n_bins']:>4} {r['strategy']:<16} "
        f"{str(r['strict']):>6} {r['acc']:>8.4f} {r['mae']:>8.4f}"
    )

best = sweep_results[0]
metrics = report(
    f"BEST — char={best['char']}, agg={best['agg']}, n_bins={best['n_bins']}, "
    f"strategy={best['strategy']}, strict={best['strict']}",
    best["y_true"], best["y_pred"], best["n_bins"],
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print("5-fold stratified CV — best config per strategy\n")
print(f"{'strategy':<16} {'char':<12} {'agg':<5} {'bins':>4} {'strict':>6} {'unique':>9}   {'Acc':>6} ± {'std':>6}   {'F1':>6} ± {'std':>6}")
print("-" * 91)

for strategy in STRATEGIES:
    b = next(r for r in sweep_results if r["strategy"] == strategy)
    features_b, outcomes_b, _, _ = feature_cache[(b["char"], b["n_bins"], b["agg"])]
    char_model_b = CHAR_MODELS[b["char"]]
    n_unique = len(np.unique(features_b, axis=0))

    fold_accs, fold_f1s = [], []
    for train_idx, test_idx in kf.split(features_b, outcomes_b):
        y_true, y_pred = run_aacbr(
            features_b[train_idx], outcomes_b[train_idx],
            features_b[test_idx],  outcomes_b[test_idx],
            char_model_b, cfg, b["n_bins"],
            strategy=strategy, strict=b["strict"],
            use_supports=False, smart_default=False, dedup=True,
        )
        fold_accs.append(float(accuracy_score(y_true, y_pred)))
        fold_f1s.append(float(f1_score(y_true, y_pred, average='macro', zero_division=0)))

    print(
        f"{strategy:<16} {b['char']:<12} {b['agg']:<5} {b['n_bins']:>4} {str(b['strict']):>6} "
        f"{f'{n_unique}/{len(features_b)}':>9}   "
        f"{np.mean(fold_accs):>6.4f} ± {np.std(fold_accs):>6.4f}   "
        f"{np.mean(fold_f1s):>6.4f} ± {np.std(fold_f1s):>6.4f}"
    )

In [ ]:
# ── log to leaderboard (optional — set SAVE_RESULTS = True to persist) ────────
SAVE_RESULTS = True
NOTES = ""  # free-text tag for this run, e.g. "after fixing slice normalisation"

if SAVE_RESULTS:
    from aacbr.results_logger import log_result
    N_BINS_RANGE = sorted({r["n_bins"] for r in sweep_results})
    for n_bins in N_BINS_RANGE:
        for strategy in STRATEGIES:
            b = next(
                (r for r in sweep_results if r["strategy"] == strategy and r["n_bins"] == n_bins),
                None,
            )
            if b is None:
                continue
            features_b, outcomes_b, train_idx_b, test_idx_b = feature_cache[(b["char"], b["n_bins"], b["agg"])]
            b_metrics = report(
                f"strategy={strategy} n_bins={n_bins} [char={b['char']}, agg={b['agg']}, strict={b['strict']}]",
                b["y_true"], b["y_pred"], b["n_bins"],
            )
            log_result({
                "eval_script": "eval_brats_gt_2d",
                "config": {
                    "char_model": b["char"],
                    "n_bins": b["n_bins"],
                    "agg_mode": b["agg"],
                    "num_slots": NUM_SLOTS,
                    "strategy": b["strategy"],
                    "strict": b["strict"],
                    "train_frac": TRAIN_FRAC,
                    "seed": SEED,
                    "resection": None,
                },
                "data_stats": {
                    "n_train": int(len(train_idx_b)),
                    "n_test": int(len(test_idx_b)),
                    "class_distribution": np.bincount(outcomes_b, minlength=b["n_bins"]).tolist(),
                },
                "metrics": b_metrics,
                "notes": NOTES,
            })